In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
base_path = "abfss://customers@adlsstdout001tgt.dfs.core.windows.net/Customers/"


In [0]:
import builtins
from pyspark.sql.functions import current_date, lit

# ---- Year ----
folders = dbutils.fs.ls(base_path)
latest_year = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Month ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/")
latest_month = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Day ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/{latest_month}/")
latest_day = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Final Path ----
latest_path = f"{base_path}{latest_year}/{latest_month}/{latest_day}/"

print(f"Reading from: {latest_path}")

# ---- File Date ----
file_date = f"{latest_year}-{latest_month}-{latest_day}"

# ---- Read parquet ----
raw_df = (
    spark.read
    .parquet(latest_path)
    .withColumn("ingest_date", current_date())
    .withColumn("file_date", lit(file_date))
)

display(raw_df.limit(10))

# Data Cleaning

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col
# =========================================================
# DATA CLEANING
# =========================================================

clean_df = (
    raw_df

    # -----------------------------------------------------
    # Remove exact duplicate rows
    # -----------------------------------------------------
    .dropDuplicates()

    # -----------------------------------------------------
    # Trim all string columns
    # -----------------------------------------------------
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_unique_id", F.trim(F.col("customer_unique_id")))
    .withColumn("customer_city", F.trim(F.col("customer_city")))
    .withColumn("customer_state", F.trim(F.col("customer_state")))

    # -----------------------------------------------------
    # Standardize text columns
    # -----------------------------------------------------
    .withColumn("customer_city", F.initcap(F.lower(F.col("customer_city"))))
    .withColumn("customer_state", F.upper(F.col("customer_state")))

    # -----------------------------------------------------
    # Clean zip code
    # Keep only numeric values
    # -----------------------------------------------------
    .withColumn(
        "customer_zip_code_prefix",
        F.regexp_replace(
            F.col("customer_zip_code_prefix").cast("string"),
            "[^0-9]",
            ""
        )
    )

    # -----------------------------------------------------
    # Handle invalid zip codes
    # Brazil ZIP prefix generally 5 digits
    # -----------------------------------------------------
    .withColumn(
        "customer_zip_code_prefix",
        F.when(
            F.length(F.col("customer_zip_code_prefix")) > 5,
            F.substring("customer_zip_code_prefix", 1, 5)
        ).otherwise(F.col("customer_zip_code_prefix"))
    )

    # -----------------------------------------------------
    # Remove rows with critical nulls
    # -----------------------------------------------------
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("customer_unique_id").isNotNull())

    # -----------------------------------------------------
    # Remove invalid customer_id records
    # UUID/hash length validation
    # -----------------------------------------------------
    .filter(F.length("customer_id") >= 32)
    .filter(F.length("customer_unique_id") >= 32)

    # -----------------------------------------------------
    # Convert dates properly
    # -----------------------------------------------------
    .withColumn("ingest_date", F.to_date("ingest_date"))
    .withColumn("file_date", F.to_date("file_date"))

    # -----------------------------------------------------
    # Add audit columns
    # -----------------------------------------------------
    .withColumn("created_date", F.current_timestamp())
)

# =========================================================
# BUSINESS LEVEL DEDUPLICATION
# Keep latest record per customer_id
# =========================================================

window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("file_date").desc(),
             F.col("ingest_date").desc())
)

final_df = (
    clean_df
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# =========================================================
# DATA QUALITY CHECKS
# =========================================================

print("Total Records :", final_df.count())

print("Null customer_id :",
      final_df.filter(F.col("customer_id").isNull()).count())

print("Duplicate customer_id :",
      final_df.groupBy("customer_id")
              .count()
              .filter("count > 1")
              .count())

In [0]:
from pyspark.sql.functions import *

new_df = (
    clean_df
    .withColumn("customer_dim_id", expr("uuid()"))
    .withColumn("is_active", lit(1))
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast("date"))
)

In [0]:

# =========================================================
# SCD TYPE 2 - DIM CUSTOMERS
# =========================================================

from pyspark.sql import functions as F

table_name = "ecom.silver.dim_silver_customers"

# =========================================================
# CHECK TABLE EXISTS
# =========================================================

if spark.catalog.tableExists(table_name):

    # =====================================================
    # CREATE TEMP VIEW
    # =====================================================

    new_df.createOrReplaceTempView("new_df")

    # =====================================================
    # STEP 1 : EXPIRE OLD RECORDS
    # =====================================================

    spark.sql("""

    MERGE INTO ecom.silver.dim_silver_customers AS t
    USING new_df AS s

    ON
        t.customer_id = s.customer_id
        AND t.is_active = 1

    WHEN MATCHED AND (

        t.customer_zip_code_prefix <> s.customer_zip_code_prefix OR
        t.customer_city <> s.customer_city OR
        t.customer_state <> s.customer_state

    )

    THEN UPDATE SET

        t.is_active = 0,
        t.end_date = current_date(),
        t.updated_timestamp = current_timestamp()

    """)

    # =====================================================
    # STEP 2 : INSERT NEW + CHANGED RECORDS
    # =====================================================

    spark.sql("""

    MERGE INTO ecom.silver.dim_silver_customers AS t
    USING new_df AS s

    ON
        t.customer_id = s.customer_id
        AND t.is_active = 1

    WHEN NOT MATCHED

    THEN INSERT (

        customer_dim_id,
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix,
        customer_city,
        customer_state,
        ingest_date,
        file_date,
        created_timestamp,
        updated_timestamp,
        is_active,
        start_date,
        end_date

    )

    VALUES (

        s.customer_dim_id,
        s.customer_id,
        s.customer_unique_id,
        s.customer_zip_code_prefix,
        s.customer_city,
        s.customer_state,
        s.ingest_date,
        s.file_date,
        current_timestamp(),
        current_timestamp(),
        1,
        current_date(),
        NULL

    )

    """)

    print("SCD Type 2 Merge Completed Successfully")

# =========================================================
# INITIAL LOAD
# =========================================================

else:

    (
        new_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option(
            "path",
            "abfss://silver@silverprocessdbstorage.dfs.core.windows.net/Customers"
        )
        .saveAsTable(table_name)
    )

    print("Initial Load Completed Successfully")